# Model Training

This notebook trains the EfficientNetV2 classification model using the TensorFlow training pipeline implemented in `src/trainer.py`.

Pipeline

1. Build model
2. Train classifier head
3. Fine-tune pretrained backbone
4. Save best model
5. Save final model
6. Export training history

In [1]:
from pathlib import Path
import sys
import shutil

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.preprocessing import DatasetPreprocessor
from src.model import ModelBuilder
from src.trainer import ModelTrainer

In [3]:
preprocessor = DatasetPreprocessor()

dataset = preprocessor.run()

2026-07-11 23:07:25 | INFO     | ======================================================================
2026-07-11 23:07:25 | INFO     | Dataset Preprocessing Initialized
2026-07-11 23:07:25 | INFO     | Dataset : C:\Users\HELGA\Documents\AI_Human_Detection\dataset_clean
2026-07-11 23:07:25 | INFO     | ======================================================================
2026-07-11 23:07:25 | INFO     | ======================================================================
2026-07-11 23:07:25 | INFO     | START PREPROCESSING
2026-07-11 23:07:25 | INFO     | ======================================================================
2026-07-11 23:07:25 | INFO     | Dataset loaded : 1525 images
2026-07-11 23:07:25 | INFO     | Train      : 1067
2026-07-11 23:07:25 | INFO     | Validation : 229
2026-07-11 23:07:25 | INFO     | Test       : 229
2026-07-11 23:07:25 | INFO     | ======================================================================
2026-07-11 23:07:25 | INFO     | DATASET STATI

In [4]:
builder = ModelBuilder()

artifacts = builder.run()

2026-07-11 23:07:26 | INFO     | ======================================================================
2026-07-11 23:07:26 | INFO     | Model Builder Initialized
2026-07-11 23:07:26 | INFO     | Model : EfficientNetV2B0
2026-07-11 23:07:26 | INFO     | ======================================================================
2026-07-11 23:07:26 | INFO     | ======================================================================
2026-07-11 23:07:26 | INFO     | START MODEL BUILDING
2026-07-11 23:07:26 | INFO     | ======================================================================
2026-07-11 23:07:29 | INFO     | Base model created.
2026-07-11 23:07:29 | INFO     | Classification head created.
2026-07-11 23:07:29 | INFO     | Model created successfully.
2026-07-11 23:07:29 | INFO     | Total Parameters : 5921874
2026-07-11 23:07:29 | INFO     | Trainable Parameters : 2562
2026-07-11 23:07:29 | INFO     | Non-trainable Parameters : 5919312
2026-07-11 23:07:29 | INFO     | ===============

In [5]:
builder.summary()

Model: "EfficientNetV2B0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,921,874 (22.59 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [6]:
trainer = ModelTrainer(
    builder=builder,
    artifacts=artifacts,
    dataset=dataset,
)

2026-07-11 23:07:29 | INFO     | ======================================================================
2026-07-11 23:07:29 | INFO     | Model Trainer Initialized
2026-07-11 23:07:29 | INFO     | ======================================================================


In [7]:
results = trainer.run(
    fine_tune=True,
)

2026-07-11 23:07:29 | INFO     | ======================================================================
2026-07-11 23:07:29 | INFO     | START TRAINING
2026-07-11 23:07:29 | INFO     | ======================================================================


2026-07-11 23:07:29 | INFO     | Model compiled (learning_rate=0.000100).
2026-07-11 23:07:29 | INFO     | 3 callbacks created.


Epoch 1/30


c:\Users\HELGA\Documents\AI_Human_Detection\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 904ms/step - accuracy: 0.5145 - loss: 0.7850
Epoch 1: val_accuracy improved from None to 0.56332, saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras

Epoch 1: finished saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.5145 - loss: 0.7850 - val_accuracy: 0.5633 - val_loss: 0.7043 - learning_rate: 1.0000e-04
Epoch 2/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - accuracy: 0.5473 - loss: 0.7381
Epoch 2: val_accuracy improved from 0.56332 to 0.58952, saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras

Epoch 2: finished saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.5473 - loss: 0.7381 - val_accuracy: 0.5895 - val_loss: 0.6677 - learning_rate: 1.0000e-04
Epoch 3/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 877ms/step - accuracy: 0.5998

2026-07-11 23:27:32 | INFO     | Initial training completed.
2026-07-11 23:27:32 | INFO     | Top 30 layers unfrozen.
2026-07-11 23:27:33 | INFO     | Model compiled (learning_rate=0.000010).
2026-07-11 23:27:33 | INFO     | 3 callbacks created.


Epoch 31/45
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7676 - loss: 0.5180
Epoch 31: val_accuracy improved from None to 0.78166, saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras

Epoch 31: finished saving model to C:\Users\HELGA\Documents\AI_Human_Detection\models\best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.7676 - loss: 0.5180 - val_accuracy: 0.7817 - val_loss: 0.5190 - learning_rate: 1.0000e-05
Epoch 32/45
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7826 - loss: 0.5045
Epoch 32: val_accuracy did not improve from 0.78166
34/34 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.7826 - loss: 0.5045 - val_accuracy: 0.7817 - val_loss: 0.5142 - learning_rate: 1.0000e-05
Epoch 33/45
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7919 - loss: 0.4923
Epoch 33: val_accuracy did not improve from 0.78166
34/34 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.7919 - loss: 0.4923 - val_accuracy: 0.7817 - val_loss: 0.5086 -

2026-07-11 23:40:07 | INFO     | Fine-tuning completed.
2026-07-11 23:40:07 | INFO     | ======================================================================
2026-07-11 23:40:07 | INFO     | TRAINING COMPLETED
2026-07-11 23:40:07 | INFO     | ======================================================================


In [8]:
trainer.export_results(results)

2026-07-11 23:40:07 | INFO     | Training history saved: C:\Users\HELGA\Documents\AI_Human_Detection\reports\metrics\training_history.csv


In [9]:
trainer.save_final_model()

2026-07-11 23:40:08 | INFO     | Final model saved: C:\Users\HELGA\Documents\AI_Human_Detection\models\final_model.keras


In [10]:
results.model

<Functional name=EfficientNetV2B0, built=True>

In [11]:
results.initial_history.history.keys()

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])

In [12]:
results.fine_tune_history.history.keys()

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])

In [13]:
history = (
    results.fine_tune_history
    if results.fine_tune_history
    else results.initial_history
)

print(history.history["accuracy"][-1])
print(history.history["val_accuracy"][-1])

0.839737594127655
0.807860255241394


## Conclusion

The training pipeline completed successfully.

- EfficientNetV2 model built.
- Initial training completed.
- Fine-tuning completed.
- Best checkpoint saved.
- Final model saved.
- Training history exported.